# Read and validate data

This notebook is designed just for convenience.

It inspects **one file per stage** (not the whole archive) and shows the result in scrollable tables:

| Stage | Location | Question |
|---|---|---|
| Raw | `data/raw/<PRODUCT>/YYYY/MM/` | Is the variable packed? Does the AOI contain valid pixels? |
| Interim | `data/interim/<PRODUCT>/YYYY/MM/` | Did crop + remap keep the grid, time steps and valid cells? |
| Zarr | `data/zarr/*.zarr` | Does the cube match the interim grid? |

Change `RAW_IDX` / `INTERIM_IDX` below to look at a different file.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import HTML, display


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find a repository containing data/")


def compute_scalar(value):
    value = value.compute() if hasattr(value, "compute") else value
    return value.item() if hasattr(value, "item") else value


def clean_text(value) -> str:
    """Make any attribute value safe for pandas/pyarrow.
    Some files carry mis-decoded bytes (lone surrogates such as 'Garc\\udcc3\\udcada'); recover them if possible."""
    text = str(value)
    try:
        text.encode("utf-8")
        return text
    except UnicodeEncodeError:
        return text.encode("utf-8", "surrogateescape").decode("utf-8", "replace")


def scroll_table(df: pd.DataFrame, height: int = 380, caption: str | None = None):
    """Show a DataFrame in a scrollable box: sticky header + sticky index, nothing truncated."""
    css = f"""<style>
    .scroll-box {{max-height:{height}px; overflow:auto; border:1px solid #8886; border-radius:4px; margin:4px 0 14px 0}}
    .scroll-box table {{border-collapse:collapse; font-size:12px; white-space:nowrap}}
    .scroll-box th, .scroll-box td {{padding:3px 10px; text-align:left; border-bottom:1px solid #8883}}
    .scroll-box thead th {{position:sticky; top:0; background:Canvas; color:CanvasText; z-index:2; border-bottom:2px solid #888}}
    .scroll-box tbody th {{position:sticky; left:0; background:Canvas; color:CanvasText; z-index:1}}
    </style>"""
    title = f"<b>{caption}</b>" if caption else ""
    table = df.to_html(na_rep="–", border=0, max_rows=None, max_cols=None,
                       float_format=lambda x: f"{x:.6g}")
    display(HTML(f"{css}{title}<div class='scroll-box'>{table}</div>"))


def open_store(path: Path, kind: str) -> xr.Dataset:
    if kind == "zarr":
        return xr.open_zarr(path, consolidated=None)
    return xr.open_dataset(path, decode_cf=True, mask_and_scale=True)


def variable_table(ds: xr.Dataset) -> pd.DataFrame:
    """One row per variable/coordinate: dtype, shape, missing values, range and on-disk packing."""
    rows = []
    for name, var in ds.variables.items():
        size = int(var.size)
        row = {"name": name,
               "role": "coord" if name in ds.coords else "data",
               "dtype": str(var.dtype),
               "dims": ", ".join(var.dims),
               "shape": " x ".join(map(str, var.shape)),
               "units": clean_text(var.attrs.get("units", "")),
               "long_name": clean_text(var.attrs.get("long_name", ""))}
        if size:
            missing = int(compute_scalar(var.isnull().sum()))
            row["missing"] = missing
            row["missing_pct"] = round(100 * missing / size, 3)
            if var.dtype.kind in "iuf" and missing < size:
                row["min"] = compute_scalar(var.min(skipna=True))
                row["max"] = compute_scalar(var.max(skipna=True))
        # packing info: is the variable stored as short/int + scale_factor/add_offset?
        enc = var.encoding
        row["on_disk_dtype"] = str(enc.get("dtype", ""))
        row["scale_factor"] = enc.get("scale_factor", "")
        row["add_offset"] = enc.get("add_offset", "")
        row["fill_value"] = enc.get("_FillValue", enc.get("fill_value", ""))
        rows.append(row)
    return pd.DataFrame(rows).set_index("name")


def attrs_table(ds: xr.Dataset) -> pd.DataFrame:
    items = {clean_text(k): clean_text(v) for k, v in ds.attrs.items() if k != "history"}
    return pd.DataFrame({"value": items}).rename_axis("attribute")


def history_table(ds: xr.Dataset) -> pd.DataFrame:
    """Split the CDO `history` attribute into one row per command (oldest first)."""
    text = clean_text(ds.attrs.get("history", "")).replace("\\n", "\n")
    rows = []
    for line in reversed([l for l in text.split("\n") if l.strip()]):
        line = re.sub(r"(/\S+)/(data|code)/", r"<repo>/\2/", line)   # shorten long paths
        m = re.match(r"^(\w{3} \w{3} +\d+ [\d:]+ \d{4}): (.*)$", line)
        rows.append({"when": m.group(1) if m else "", "command": m.group(2) if m else line})
    return pd.DataFrame(rows).rename_axis("step")


def grid_summary(ds: xr.Dataset, vt: pd.DataFrame) -> dict:
    var = "METREF" if "METREF" in ds.data_vars else next(
        n for n, d in ds.data_vars.items() if d.dtype.kind in "iuf")
    lat_name = next(n for n in ("lat", "latitude", "y") if n in ds.coords)
    lon_name = next(n for n in ("lon", "longitude", "x") if n in ds.coords)
    lat, lon = ds[lat_name].values, ds[lon_name].values
    step = lambda a: round(float(abs(np.diff(a)).mean()), 4) if a.size > 1 else ""
    return {"variable": var,
            "time steps": ds.sizes.get("time", 1),
            "grid (lat x lon)": f"{lat.size} x {lon.size}",
            "lat range": f"{lat.min():.3f} .. {lat.max():.3f}",
            "lon range": f"{lon.min():.3f} .. {lon.max():.3f}",
            "lat step": step(lat), "lon step": step(lon),
            "valid cells %": round(100 - float(vt.loc[var, "missing_pct"]), 2)}


def inspect_one(kind: str, path: Path):
    """Open ONE file/store, show its tables, return (variable table, grid summary)."""
    print(f"\n{kind.upper()}: {path.relative_to(REPO_ROOT)}")
    with open_store(path, kind) as ds:
        print("Dimensions:", dict(ds.sizes))
        vt = variable_table(ds)
        scroll_table(vt, caption="Variables (scroll sideways for packing info)")
        scroll_table(history_table(ds), height=220, caption="Processing history (oldest -> newest)")
        scroll_table(attrs_table(ds), height=260, caption="Global attributes")
        return vt, grid_summary(ds, vt)


REPO_ROOT = find_repo_root()
DATA_ROOT = REPO_ROOT / "data"
raw_files = sorted((DATA_ROOT / "raw").rglob("*.nc"))
interim_files = sorted((DATA_ROOT / "interim").rglob("*.nc"))
zarr_stores = sorted(p for p in (DATA_ROOT / "zarr").iterdir()
                     if p.is_dir() and ((p / "zarr.json").exists() or (p / ".zgroup").exists()))
print(f"Found {len(raw_files)} raw, {len(interim_files)} interim, and {len(zarr_stores)} Zarr stores.")

## which single file to inspect per stage (index into the sorted lists above)
RAW_IDX, INTERIM_IDX, ZARR_IDX = 0, 0, 0

tables, summaries = {}, {}

Found 31 raw, 31 interim, and 2 Zarr stores.


## RAW data

In [2]:
tables["RAW"], summaries["RAW"] = inspect_one("raw", raw_files[RAW_IDX])


RAW: data/raw/METREF/2010/01/NETCDF4_LSASAF_MSG_METREF_MSG-Disk_201001010000.nc
Dimensions: {'time': 1, 'lat': 3201, 'lon': 3201}


,role,dtype,dims,shape,units,long_name,missing,missing_pct,min,max,on_disk_dtype,scale_factor,add_offset,fill_value
name,,,,,,,,,,,,,,
METREF,data,float64,"time, lat, lon",1 x 3201 x 3201,mm/day,METREF,7561755,73.799,0,6.96,int32,0.01,,-8000
crs,data,|S1,,,,,0,0,–,–,|S1,,,
quality_flag,data,float64,"time, lat, lon",1 x 3201 x 3201,,QFLAGS,2204174,21.512,-4,6,int32,1,,-1
lat,coord,float32,lat,3201,degrees_north,latitude,0,0,-80,80,float32,,,
lon,coord,float32,lon,3201,degrees_east,longitude,0,0,-80,80,float32,,,
time,coord,datetime64[ns],time,1,,time,0,0,–,–,int16,,,


,when,command
step,,
0,,2020-11-09T17:30:27Z - NetCDF reprojected product ; 2019-02-15T21:21:21Z Original product file created: HDF5_LSASAF_MSG_METREF_MSG-Disk_201001010000


,value
attribute,
date_created,2020-11-09T17:30:27Z
algorithm_version,1.3.2
base_algorithm_version,1.0.3
processing_level,03
platform,MSG2
sensor,SEVI
parent_product_name,METREF;QFLAGS;-;-
time_coverage_start,2010-01-01T00:00:00Z
image_reference_time,2010-01-01T00:00:00Z


## INTERIM data

In [3]:
tables["INTERIM"], summaries["INTERIM"] = inspect_one("interim", interim_files[INTERIM_IDX])


INTERIM: data/interim/METREF/2010/01/NETCDF4_LSASAF_MSG_METREF_MSG-Disk_201001010000_processed.nc
Dimensions: {'time': 1, 'lat': 91, 'lon': 121}


,role,dtype,dims,shape,units,long_name,missing,missing_pct,min,max,on_disk_dtype,scale_factor,add_offset,fill_value
name,,,,,,,,,,,,,,
METREF,data,float32,"time, lat, lon",1 x 91 x 121,mm/day,METREF,1039,9.436,2.91001,3.68,float32,,,-8000
quality_flag,data,float64,"time, lat, lon",1 x 91 x 121,,QFLAGS,0,0,0,1,int32,,,-1
time,coord,datetime64[ns],time,1,,time,0,0,–,–,float64,,,
lon,coord,float64,lon,121,degrees_east,longitude,0,0,-17.25,-11.25,float64,,,
lat,coord,float64,lat,91,degrees_north,latitude,0,0,12.25,16.75,float64,,,


,when,command
step,,
0,,2020-11-09T17:30:27Z - NetCDF reprojected product ; 2019-02-15T21:21:21Z Original product file created: HDF5_LSASAF_MSG_METREF_MSG-Disk_201001010000
1,Fri Oct 16 00:43:18 2026,"cdo -L -f nc4 -O -sellonlatbox,-17.475,-11.025,12.075,16.925 data/raw/METREF/2010/01/NETCDF4_LSASAF_MSG_METREF_MSG-Disk_201001010000.nc data/interim/METREF/2010/01/.clip.tuLKzJ.nc"
2,Fri Oct 16 00:43:18 2026,"cdo -L -f nc4 -O -delname,quality_flag data/interim/METREF/2010/01/.clip.tuLKzJ.nc data/interim/METREF/2010/01/.main.y0XaVh.nc"
3,Fri Oct 16 00:43:18 2026,"cdo -L -f nc4 -O -remapbil,<repo>/code/preprocess_utils/grid.txt data/interim/METREF/2010/01/.main.y0XaVh.nc data/interim/METREF/2010/01/.main_remap.PeXTzu.nc"
4,Fri Oct 16 00:43:18 2026,cdo -L -f nc4 -O merge data/interim/METREF/2010/01/.main_remap.PeXTzu.nc data/interim/METREF/2010/01/.qa_remap.wBW9HG.nc data/interim/METREF/2010/01/.merged.879lX1.nc


,value
attribute,
CDI,Climate Data Interface version 2.4.0 (https://mpimet.mpg.de/cdi)
Conventions,CF-1.6
institution,IM-PT
date_created,2020-11-09T17:30:27Z
algorithm_version,1.3.2
base_algorithm_version,1.0.3
processing_level,03
platform,MSG2
sensor,SEVI


## ZARR data

In [4]:
tables["ZARR"], summaries["ZARR"] = inspect_one("zarr", zarr_stores[ZARR_IDX])


ZARR: data/zarr/metref_cube.zarr
Dimensions: {'time': 31, 'lat': 91, 'lon': 121}


,role,dtype,dims,shape,units,long_name,missing,missing_pct,min,max,on_disk_dtype,scale_factor,add_offset,fill_value
name,,,,,,,,,,,,,,
METREF,data,float32,"time, lat, lon",31 x 91 x 121,mm/day,METREF,32209,9.436,1.35001,4.21,float32,,,-8000
quality_flag,data,float64,"time, lat, lon",31 x 91 x 121,,QFLAGS,0,0,0,2,int32,,,-1
lat,coord,float64,lat,91,degrees_north,latitude,0,0,12.25,16.75,float64,,,–
lon,coord,float64,lon,121,degrees_east,longitude,0,0,-17.25,-11.25,float64,,,–
time,coord,datetime64[ns],time,31,,time,0,0,–,–,float64,,,–


,when,command
step,,
0,,2020-11-09T17:30:27Z - NetCDF reprojected product ; 2019-02-15T21:21:21Z Original product file created: HDF5_LSASAF_MSG_METREF_MSG-Disk_201001010000
1,Fri Oct 16 00:43:18 2026,"cdo -L -f nc4 -O -sellonlatbox,-17.475,-11.025,12.075,16.925 data/raw/METREF/2010/01/NETCDF4_LSASAF_MSG_METREF_MSG-Disk_201001010000.nc data/interim/METREF/2010/01/.clip.tuLKzJ.nc"
2,Fri Oct 16 00:43:18 2026,"cdo -L -f nc4 -O -delname,quality_flag data/interim/METREF/2010/01/.clip.tuLKzJ.nc data/interim/METREF/2010/01/.main.y0XaVh.nc"
3,Fri Oct 16 00:43:18 2026,"cdo -L -f nc4 -O -remapbil,<repo>/code/preprocess_utils/grid.txt data/interim/METREF/2010/01/.main.y0XaVh.nc data/interim/METREF/2010/01/.main_remap.PeXTzu.nc"
4,Fri Oct 16 00:43:18 2026,cdo -L -f nc4 -O merge data/interim/METREF/2010/01/.main_remap.PeXTzu.nc data/interim/METREF/2010/01/.qa_remap.wBW9HG.nc data/interim/METREF/2010/01/.merged.879lX1.nc


,value
attribute,
CDI,Climate Data Interface version 2.4.0 (https://mpimet.mpg.de/cdi)
Conventions,CF-1.6
institution,IM-PT
date_created,2020-11-09T17:30:27Z
algorithm_version,1.3.2
base_algorithm_version,1.0.3
processing_level,03
platform,MSG2
sensor,SEVI


## Stage comparison

In [5]:
scroll_table(pd.DataFrame(summaries), height=300, caption="Raw vs interim vs Zarr (one file each)")

if summaries["INTERIM"]["grid (lat x lon)"] != summaries["ZARR"]["grid (lat x lon)"]:
    print("WARNING: the Zarr grid differs from the interim grid - the Zarr was probably built "
          "from older interim files and needs to be rebuilt.")

,RAW,INTERIM,ZARR
variable,METREF,METREF,METREF
time steps,1,1,31
grid (lat x lon),3201 x 3201,91 x 121,91 x 121
lat range,-80.000 .. 80.000,12.250 .. 16.750,12.250 .. 16.750
lon range,-80.000 .. 80.000,-17.250 .. -11.250,-17.250 .. -11.250
lat step,0.05,0.05,0.05
lon step,0.05,0.05,0.05
valid cells %,26.2,90.56,90.56


## Missing-value comparison

In [6]:
combined = pd.concat(tables, names=["stage", "name"]).reset_index()
cols = ["stage", "name", "role", "missing", "missing_pct", "dtype", "shape"]
has_missing = combined.loc[combined["missing"] > 0, cols].set_index(["stage", "name"])
scroll_table(has_missing, caption="Variables with missing values")